# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv("data/AviationData.csv", encoding="latin1", low_memory=False)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


- Rows: accident record
- Columns: accident variable

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

### Getting Info
Observing column and df info to prepare for filtering.
- Year: 1983 and  onwards
- Build: professional only

In [3]:
print("Column Names:")
print(df.columns, "\n")    # column names

print("Dataframe Info:")
df.info()                  # display data types and missing value

print("\nSummary Statistics:") 
df.describe()              # summary 

Column Names:
Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object') 

Dataframe Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Numbe

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


#### Filtering by Date
Filter out events before 1983

In [4]:
# make event
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# filter out events before 1983
df = df[df['Event.Date'].dt.year >= 1983]

# updated df shape
df.shape

(85289, 31)

After converting `Event.Date` to a proper format and filtering, the data shows 85,289 records of incidents that occured after 1983 with 31 columns.

#### Filtering by Aircraft Build (Professional/Amateur)
Filter out aircrafts that are amateur, client only interested in professional builds

In [5]:
df = df[df['Amateur.Built'] != 'Yes']        # filter rows where aircraft build is amateur
df['Amateur.Built'].value_counts(dropna=False)  # count how many aircrafts are amateur

Amateur.Built
No     76960
NaN      101
Name: count, dtype: int64

After filtering, data shows 76,960 professional aircrafts and 101 with unknown build.

#### Inspecting Aircraft Columns
Inspect the columns relevant to aircraft characteristics to plan for imputation or any more filtering

In [6]:
# define relevant columms to client needs
relevant_cols = ['Make', 'Model', 'Aircraft.Category', 'Number.of.Engines', 
                 'Engine.Type', 'Schedule', 'Purpose.of.flight']

print("Relevant Column .info():")
df[relevant_cols].info()  # show datatype & missing info

print("\nRelevant Column .head():")
df[relevant_cols].head()  # show first 5 rows of cols

Relevant Column .info():
<class 'pandas.core.frame.DataFrame'>
Index: 77061 entries, 3600 to 88888
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Make               77013 non-null  object 
 1   Model              76992 non-null  object 
 2   Aircraft.Category  25423 non-null  object 
 3   Number.of.Engines  71274 non-null  float64
 4   Engine.Type        70299 non-null  object 
 5   Schedule           11808 non-null  object 
 6   Purpose.of.flight  70930 non-null  object 
dtypes: float64(1), object(6)
memory usage: 4.7+ MB

Relevant Column .head():


,Make,Model,Aircraft.Category,Number.of.Engines,Engine.Type,Schedule,Purpose.of.flight
3600,Piccard,AX-6,NaN,NaN,Unknown,NaN,Personal
3601,Cessna,182P,NaN,1.0,Reciprocating,NaN,Personal
3602,Cessna,182RG,NaN,1.0,Reciprocating,NaN,Personal
3603,Cessna,182P,NaN,1.0,Reciprocating,NaN,Personal
3604,Piper,PA-28R-200,NaN,1.0,Reciprocating,NaN,Personal


- Make & Model almost complete
- Aircraft.Category missing ~66% of data
- Number.of.Engines & Engine.Type mostly complete
- Schedule mostly missing ~86% of data
- Purpose.of.flight mostly complete

#### Handle Missing Data

Fill missing `Aircraft.Category` with 'Unknown' and drop rows missing `Make` or `Model` to ensure we have valid data for analysis

In [7]:
# fill missing category w/ 'Unknown'
df['Aircraft.Category'] = df['Aircraft.Category'].fillna('Unknown')

df = df.dropna(subset=['Make', 'Model'])  # drop rows where make or model missing
df.shape #updated df

(76986, 31)

After replacing missing `Aircraft.Category` values with 'Unknown' and dropping missing `Make` and `Model` rows, 76,986 entries with 31 colunmns remain

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [8]:
# missing injury values fill in with 0
df['Total.Fatal.Injuries'] = df['Total.Fatal.Injuries'].fillna(0)
df['Total.Serious.Injuries'] = df['Total.Serious.Injuries'].fillna(0)
df['Total.Minor.Injuries'] = df['Total.Minor.Injuries'].fillna(0)
df['Total.Uninjured'] = df['Total.Uninjured'].fillna(0)

# Constructing Metric for Fatal/Serious Injuries
# make new column 'Total.Passengers' by adding all injuries to estimate total passengers
df['Total.Passengers'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + \
                         df['Total.Minor.Injuries'] + df['Total.Uninjured']
# metric: fraction of passengers w/ fatal or serious injury
# (fatal+serious) / total
# if Total.Passengers is 0, replace with NaN to avoid dividing by 0
df['Frac.Fatal.Serious'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers'].replace(0, np.nan)

#check
df[['Total.Passengers','Frac.Fatal.Serious', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']].head()


,Total.Passengers,Frac.Fatal.Serious,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
3600,2.0,0.5,0.0,1.0,0.0,1.0
3601,4.0,0.0,0.0,0.0,1.0,3.0
3602,2.0,0.0,0.0,0.0,0.0,2.0
3603,1.0,0.0,0.0,0.0,0.0,1.0
3604,2.0,0.0,0.0,0.0,2.0,0.0


- Missing injury values (NaN) assumed to mean no injuries, so replaced with 0
- Total.Passengers = sum of fatal, serious, minor, and uninjured passengers -> assumed to be total based on injured+non-injured
- Frac.Fatal.Serious = frac of passengers with fatal or serious injuries
- Dividing by 0: to prevent issues with dividing over 0 if Total.Passengers is 0, those entries are set to NaN 

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [9]:
# getting values in Aircraft.damage to inspect
print("Value Counts:")
print(df['Aircraft.damage'].value_counts(dropna=False))

# Cleanning
# replace missing vals with 'Unknown'
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('Unknown')

# Derived Column
# if damage indicates Destroyed; set to True
df['Destroyed'] = df['Aircraft.damage'].apply(lambda x: True if x.lower() == 'destroyed' else False)

df[['Aircraft.damage','Destroyed']].head(10)   #check

Value Counts:
Aircraft.damage
Substantial    55689
Destroyed      15489
NaN             3088
Minor           2604
Unknown          116
Name: count, dtype: int64


,Aircraft.damage,Destroyed
3600,Unknown,False
3601,Substantial,False
3602,Substantial,False
3603,Substantial,False
3604,Substantial,False
3605,Substantial,False
3606,Unknown,False
3607,Destroyed,True
3608,Destroyed,True
3609,Destroyed,True


- Missing vals in `Aircraft.damange` are replaced with 'Unknown' because we cannot assume damage if not reported
- Derived column `Destroyed` is True if damage indicated destroyed, or False otherwise
    - Column aids in further analysis

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

#### Cleaning Tasks
1. Fill missing vals with 'Unknown' (we can't know/assume the make if its missing)
2. Standardize caps to title case (aesthetics/cleanliness)
3. Remove typos/invalid entries manually
4. Keep only makes with >= 50 occurences 

In [10]:
# get values in Make column to inspect
print("Make Value Counts (Top 20):")
print(df['Make'].value_counts(dropna=False).head(20)) # before

# Cleaning
df['Make'] = df['Make'].fillna('Unknown')  # fill missing
df['Make'] = df['Make'].str.title()        # standardize capitalization

# filter to keep Makes with 50 occurrences or more
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print("\nRemaining Makes:", df['Make'].nunique())
df['Make'].value_counts().head(20)         # check, after


Make Value Counts (Top 20):
Make
Cessna               20835
Piper                11268
CESSNA                4919
Beech                 4056
PIPER                 2838
Bell                  2007
Boeing                1530
BOEING                1145
BEECH                 1041
Mooney                1029
Grumman                987
Robinson               915
Bellanca               819
Hughes                 740
Schweizer              610
BELL                   588
Air Tractor            577
Mcdonnell Douglas      517
Aeronca                458
Maule                  427
Name: count, dtype: int64

Remaining Makes: 90


Make
Cessna               25754
Piper                14106
Beech                 5097
Boeing                2675
Bell                  2595
Mooney                1271
Robinson              1197
Grumman               1064
Bellanca               978
Hughes                 877
Schweizer              754
Air Tractor            673
Aeronca                607
Mcdonnell Douglas      599
Maule                  571
Champion               504
Stinson                421
Aero Commander         411
De Havilland           404
Luscombe               390
Name: count, dtype: int64

After cleaning and filtering, theres 90 unique aircraft Makes in the dataset that meet the threshhold w/ a sufficient sample size

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [11]:
# get vals in Model column to inspect
print("Model Value Counts (Top 20):")
print(df['Model'].value_counts(dropna=False).head(20))  # before

# Cleaning
df = df.dropna(subset=['Model'])      # drop rows where Model missing
df['Model'] = df['Model'].str.title() # standardize capitalization

# Check if models unique within each Make
model_make_counts = df.groupby(['Make','Model']).size().reset_index(name='Count')
duplicate_models = model_make_counts.groupby('Model')['Make'].nunique()
non_unique_models = duplicate_models[duplicate_models > 1]

print("\nNon-unique Models across Makes:")
print(non_unique_models)          # models that appear under more than one Make

# Derived Column
# combine Make & Model to create unique plane type identifier
df['Plane.Type'] = df['Make'] + " " + df['Model']

# Check
df[['Make','Model','Plane.Type']].head(10)


Model Value Counts (Top 20):
Model
152          2229
172          1650
172N         1093
PA-28-140     865
172M          759
150           752
172P          665
182           617
180           596
150M          551
PA-18         550
PA-18-150     549
PA-28-180     547
PA-28-161     523
PA-28-181     509
737           488
206B          488
150L          435
A36           428
PA-38-112     424
Name: count, dtype: int64

Non-unique Models across Makes:
Model
100                  4
100-180              2
105                  2
109                  2
109A                 2
                    ..
Tb-9                 2
Tb20                 2
Tb21                 2
Thrush S2R           3
Tiger Moth Dh 82A    2
Name: Make, Length: 467, dtype: int64


,Make,Model,Plane.Type
3601,Cessna,182P,Cessna 182P
3602,Cessna,182Rg,Cessna 182Rg
3603,Cessna,182P,Cessna 182P
3604,Piper,Pa-28R-200,Piper Pa-28R-200
3605,Cessna,140,Cessna 140
3606,Balloon Works,Firefly 7B,Balloon Works Firefly 7B
3607,Cessna,340A,Cessna 340A
3608,North American,T-6G,North American T-6G
3609,Piper,Pa-24-250,Piper Pa-24-250
3610,Piper,Pa-32-301R,Piper Pa-32-301R


- Dropped rows missing `Model` (cannot assume)
- Standardized capitilization for consistency
- Derived Column: Plane.Type = Make + Model to make unique aircraft type identifier

Each `Plane.Type` can identify specific aircraft type for analysis

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [12]:
# getting columns to inspect
cols_to_check = ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 
                 'Purpose.of.flight', 'Broad.phase.of.flight']

# inspect types and missing values
df[cols_to_check].info()

# inspecting top val counts for each column
for col in cols_to_check:
    print(f"\nValue counts for {col} (Top 10):")
    print(df[col].value_counts(dropna=False).head(10))

# Cleaning
# standardize str cols to title case to reduce duplicates due to case sensitivity
# leave out Number.of.Engines, not strings
str_cols = ['Engine.Type','Weather.Condition','Purpose.of.flight','Broad.phase.of.flight']
for col in str_cols:
    df[col] = df[col].str.title()

# check
for col in cols_to_check:
    print(f"\nCleaned Top Values for {col}:")
    print(df[col].value_counts(dropna=False).head(10))

<class 'pandas.core.frame.DataFrame'>
Index: 70539 entries, 3601 to 88888
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Engine.Type            65049 non-null  object 
 1   Weather.Condition      66788 non-null  object 
 2   Number.of.Engines      65643 non-null  float64
 3   Purpose.of.flight      65150 non-null  object 
 4   Broad.phase.of.flight  50061 non-null  object 
dtypes: float64(1), object(4)
memory usage: 3.2+ MB

Value counts for Engine.Type (Top 10):
Engine.Type
Reciprocating      55014
NaN                 5490
Turbo Shaft         2989
Turbo Prop          2796
Turbo Fan           2178
Unknown             1514
Turbo Jet            543
Geared Turbofan       12
LR                     1
UNK                    1
Name: count, dtype: int64

Value counts for Weather.Condition (Top 10):
Weather.Condition
VMC    60586
IMC     5269
NaN     3751
UNK      704
Unk      229
Name: count, dtype: i

- `Engine.Type`, `Weather.Condition`, `Purpose.of.flight`, `Broad.phase.of.flight`: Standardized data to title case
- `Number.of.Engines`: left numeric, left missing vals unfilled
- No dropped rows, missing vals retained

Cols now consistent

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [13]:
# Inspect overall missing values as a percentage
missing_pct = df.isna().mean() * 100
missing_pct.sort_values(ascending=False)

Schedule                  84.582997
Air.carrier               81.199053
FAR.Description           69.725967
Longitude                 63.176399
Latitude                  63.169311
Airport.Code              43.556047
Airport.Name              40.931967
Broad.phase.of.flight     29.030749
Publication.Date          17.292562
Engine.Type                7.782929
Purpose.of.flight          7.639745
Report.Status              7.161995
Number.of.Engines          6.940841
Weather.Condition          5.317626
Registration.Number        1.635974
Frac.Fatal.Serious         1.613292
Injury.Severity            1.247537
Country                    0.275025
Amateur.Built              0.123336
Location                   0.068047
Total.Minor.Injuries       0.000000
Total.Passengers           0.000000
Destroyed                  0.000000
Total.Uninjured            0.000000
Event.Id                   0.000000
Total.Serious.Injuries     0.000000
Total.Fatal.Injuries       0.000000
Investigation.Type         0

After calculating % missing vals for each col; `Schedule`, `Air.carrier`, `FAR.Description`, `Longitude`, and `Latitude` have over 50% mising 

Therefore, these columns may not be reliable for analysis

Key Columns:`Make`, `Model`, `Aircraft.Category`, `Total.Fatal.Injuries`, `Total.Serious.Injuries`, `Total.Passengers`, and `Destroyed` have few missing values

**Conclusion**: Since key attributes have little to no missing values, the dataset is mostly complete to analyze

In [14]:
# identify columns w/ > 50% missing values
cols_to_drop = missing_pct[missing_pct > 50].index

# drop cols from dataframe
df = df.drop(columns=cols_to_drop)

df.info() #check

<class 'pandas.core.frame.DataFrame'>
Index: 70539 entries, 3601 to 88888
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                70539 non-null  object        
 1   Investigation.Type      70539 non-null  object        
 2   Accident.Number         70539 non-null  object        
 3   Event.Date              70539 non-null  datetime64[ns]
 4   Location                70491 non-null  object        
 5   Country                 70345 non-null  object        
 6   Airport.Code            39815 non-null  object        
 7   Airport.Name            41666 non-null  object        
 8   Injury.Severity         69659 non-null  object        
 9   Aircraft.damage         70539 non-null  object        
 10  Aircraft.Category       70539 non-null  object        
 11  Registration.Number     69385 non-null  object        
 12  Make                    70539 non-null  object  

- Dropped cols with **> 50% missing vals**: `Schedule`, `Air.carrier`, `FAR.Description`, `Longitude`, `Latitude`
- Cols with moderate missing values were kept for analysis, as they may still provide useful insights

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [15]:
df.to_csv("data/AviationData_Cleaned.csv", index=False)